# revT2V: zero-shot Attention Rotation test (v2)

**Question:** if we rotate ModelScope's temporal self-attention maps by 180° with **no training**, does the generated motion run backward?

ModelScope has **two** temporal self-attentions per temporal block (`attn1` + `attn2`, the paper's N₄ = 2) plus `transformer_in`: **34 temporal attention layers** in total. v1 only rotated `attn1`. This version compares:

| Variant | Rotated layers | Expected count |
|---|---|---|
| `normal` | none | 0 |
| `rot_up` | up-block temporal `attn1` only (v1 setting) | 9 |
| `rot_up_both` | up-block temporal `attn1` + `attn2` | 18 |
| `rot_blocks_both` | all block temporal attns, not `transformer_in` | 32 |
| `rot_all_both` | every temporal attention | 34 |

Same seed per prompt across variants, so differences come only from the rotation. Motion direction is measured with optical flow.

Runtime: T4 GPU, ~20 s per video. Default run = 3 prompts × 3 seeds × 5 variants ≈ 15-20 min.

## 1. Setup

In [ ]:
# Change this to your GitHub username
GITHUB_USER = "<your-github-username>"

!git clone -q https://github.com/{GITHUB_USER}/revT2V.git /content/revT2V 2>/dev/null || git -C /content/revT2V pull -q
%cd /content/revT2V
!pip install -q -e .
# no %autoreload: it breaks on Colab's Python 3.13 (removed `imp` module).
# After a git pull, restart the kernel and re-run from the top.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 2. Load the teacher

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from diffusers.models.attention_processor import AttnProcessor, AttnProcessor2_0
from diffusers.utils import export_to_video
from IPython.display import Video, display

from revt2v.teacher import load_teacher
from revt2v.methods.attn_rotation import RotatedTemporalAttnProcessor   # adjust if your class lives elsewhere

pipe = load_teacher()
pipe.set_progress_bar_config(disable=True)

print("prediction_type:", pipe.scheduler.config.prediction_type)
print("text hidden size:", pipe.text_encoder.config.hidden_size)
print("unet frozen:", not any(p.requires_grad for p in pipe.unet.parameters()))

## 3. Find the temporal attention layers

In [ ]:
# select temporal layers by CLASS, not by name: catches transformer_in too
temporal_prefixes = [n for n, m in pipe.unet.named_modules()
                     if type(m).__name__ == "TransformerTemporalModel"]

def is_temporal(name):
    return any(name.startswith(p + ".") for p in temporal_prefixes)

temporal = [n for n in pipe.unet.attn_processors if is_temporal(n)]
print(f"{len(pipe.unet.attn_processors)} attention layers total, {len(temporal)} temporal (expected 34)\n")

for n in temporal:
    m = pipe.unet.get_submodule(n.rsplit(".processor", 1)[0])
    print(f"{n:75s} | {'cross' if m.is_cross_attention else 'self '} | dim {m.to_q.in_features}")

## 4. Sanity check: is the rotated processor correct?

For one layer of each kind, compare your processor with a reference (stock attention with its F×F map explicitly rotated 180°).
`ours-ref` should be ~0 (exactly 0 is fine) and `ours-stock` clearly larger.

In [ ]:
@torch.no_grad()
def reference_rotated(attn, h):
    q = attn.head_to_batch_dim(attn.to_q(h))
    k = attn.head_to_batch_dim(attn.to_k(h))
    v = attn.head_to_batch_dim(attn.to_v(h))
    probs = attn.get_attention_scores(q, k, None)      # (pixels*heads, F, F)
    probs = probs.flip(-1).flip(-2)                     # rotate 180°
    out = attn.batch_to_head_dim(torch.bmm(probs, v))   # merge heads
    out = attn.to_out[1](attn.to_out[0](out))           # output projection + dropout
    if attn.residual_connection:
        out = out + h
    return out / attn.rescale_output_factor

@torch.no_grad()
def check(attn_name, num_pixels=4, num_frames=16):
    attn = pipe.unet.get_submodule(attn_name.rsplit(".processor", 1)[0])
    x = torch.randn(num_pixels, num_frames, attn.to_q.in_features, device="cuda", dtype=pipe.unet.dtype)
    ours, ref, stock = RotatedTemporalAttnProcessor()(attn, x), reference_rotated(attn, x), AttnProcessor()(attn, x)
    d_ref, d_stock = (ours - ref).abs().max().item(), (ours - stock).abs().max().item()
    ok = d_ref < 1e-2 and d_stock > 10 * max(d_ref, 1e-4)
    print(f"{'OK  ' if ok else 'FAIL'} {attn_name}\n      ours-ref: {d_ref:.2e}   ours-stock: {d_stock:.2e}")
    return ok

to_test = [
    next(n for n in temporal if "temp_attentions" in n and "attn1" in n),
    next(n for n in temporal if "temp_attentions" in n and "attn2" in n),
    next(n for n in temporal if "transformer_in" in n and "attn1" in n),
    next(n for n in temporal if "transformer_in" in n and "attn2" in n),
]
results = [check(n) for n in to_test]
print("\nALL PASSED" if all(results) else "\nSOMETHING FAILED: fix the processor before running the experiment")

## 5. Variants and helpers

In [ ]:
VARIANTS = {
    "normal":          None,
    "rot_up":          lambda n: n.startswith("up_blocks") and is_temporal(n) and "attn1" in n,
    "rot_up_both":     lambda n: n.startswith("up_blocks") and is_temporal(n),
    "rot_blocks_both": lambda n: is_temporal(n) and not n.startswith("transformer_in"),
    "rot_all_both":    is_temporal,
}
EXPECTED = {"normal": 0, "rot_up": 9, "rot_up_both": 18, "rot_blocks_both": 32, "rot_all_both": 34}

def apply_variant(name):
    filt = VARIANTS[name]
    if filt is None:
        pipe.unet.set_attn_processor(AttnProcessor2_0())
    else:
        pipe.unet.set_attn_processor({
            n: (RotatedTemporalAttnProcessor() if filt(n) else AttnProcessor2_0())
            for n in pipe.unet.attn_processors
        })
    k = sum(isinstance(p, RotatedTemporalAttnProcessor) for p in pipe.unet.attn_processors.values())
    assert k == EXPECTED[name], f"{name}: rotated {k} layers, expected {EXPECTED[name]}"
    return k

# verify every variant selects the right layers BEFORE spending GPU time
for v in VARIANTS:
    print(f"{v:16s} -> {apply_variant(v)} layers rotated")
apply_variant("normal")

NEGATIVE = "watermark, text"   # reduces the shutterstock watermark; same for every variant

@torch.no_grad()
def gen(prompt, seed=0, steps=25):
    g = torch.Generator("cuda").manual_seed(seed)
    v = pipe(prompt, negative_prompt=NEGATIVE, num_frames=16, height=256, width=256,
             num_inference_steps=steps, generator=g, output_type="np").frames[0]   # (16, 256, 256, 3) in [0,1]
    return (np.clip(v, 0, 1) * 255).astype(np.uint8)

def flow_stats(video):
    """Dominant motion of a (F, H, W, 3) uint8 video.
    Returns mean flow magnitude and mean (dx, dy) of the 10% fastest-moving pixels."""
    gray = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in video]
    mags, dxs, dys = [], [], []
    for a, b in zip(gray[:-1], gray[1:]):
        fl = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        m = np.linalg.norm(fl, axis=2)
        top = m > np.percentile(m, 90)
        mags.append(m.mean()); dxs.append(fl[..., 0][top].mean()); dys.append(fl[..., 1][top].mean())
    return float(np.mean(mags)), float(np.mean(dxs)), float(np.mean(dys))

def side_by_side(videos):
    return np.concatenate(videos, axis=2)   # (F, H, W*len, 3)

## 6. Run the experiment

In [ ]:
OUT = "results/zero_shot_rotation_v2"
os.makedirs(OUT, exist_ok=True)

prompts = [
    "a red car driving to the right on a road",
    "a person walking forward on a beach",
    "a glass falling off a table and shattering on the floor",
]
SEEDS = [0, 1, 2]          # fewer seeds = faster; more seeds = more trustworthy
SHOW_SEED = 0              # which seed to display inline

rows = []
for i, p in enumerate(prompts):
    for seed in SEEDS:
        print(f"[{i}] seed {seed}: {p}")
        vids = []
        for name in VARIANTS:
            apply_variant(name)
            v = gen(p, seed=seed)
            mag, dx, dy = flow_stats(v)
            rows.append(dict(prompt=p, seed=seed, variant=name, motion=mag, dx=dx, dy=dy))
            export_to_video(list(v), f"{OUT}/p{i}_s{seed}_{name}.mp4", fps=8)
            vids.append(v)
        path = f"{OUT}/p{i}_s{seed}_compare.mp4"
        export_to_video(list(side_by_side(vids)), path, fps=8)
        if seed == SHOW_SEED:
            print("   left -> right:", " | ".join(VARIANTS))
            display(Video(path, embed=True, width=256 * len(VARIANTS)))

apply_variant("normal")
torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/flow_metrics.csv", index=False)
print("saved", f"{OUT}/flow_metrics.csv")

## 7. Did the motion reverse?

For each prompt + seed, compare each variant's dominant motion direction with `normal`'s:
- **cos** = cosine similarity between the (dx, dy) vectors: **−1 means perfectly reversed**, +1 same direction.
- **motion_ratio** = variant motion / normal motion: ~1 keeps motion, near 0 means frozen.

A variant "reverses" when cos < −0.5 and it still moves (motion_ratio > 0.3).

In [ ]:
base = df[df.variant == "normal"].set_index(["prompt", "seed"])

def compare(r):
    b = base.loc[(r.prompt, r.seed)]
    v1, v2 = np.array([b.dx, b.dy]), np.array([r.dx, r.dy])
    cos = float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8))
    return pd.Series(dict(cos=cos, motion_ratio=r.motion / (b.motion + 1e-8)))

cmp = df[df.variant != "normal"].copy()
cmp[["cos", "motion_ratio"]] = cmp.apply(compare, axis=1)
cmp["reversed"] = (cmp.cos < -0.5) & (cmp.motion_ratio > 0.3)
cmp["frozen"] = cmp.motion_ratio < 0.3
cmp.to_csv(f"{OUT}/comparison.csv", index=False)

print("Per prompt / seed:")
display(cmp[["prompt", "seed", "variant", "cos", "motion_ratio", "reversed", "frozen"]].round(2))

print("\nSummary per variant (fractions over all prompts x seeds):")
display(cmp.groupby("variant", sort=False)[["reversed", "frozen", "motion_ratio", "cos"]].mean().round(2))

## 8. Notes for the report

- Which variant reverses most often **and** still moves?
- Does adding `attn2` (`rot_up` vs `rot_up_both`) help or hurt?
- Does `transformer_in` matter (`rot_blocks_both` vs `rot_all_both`)?
- Quality: does the subject survive (car/person visible), or does it vanish?

Caveat: optical flow on generated video is noisy, especially when the normal video's motion is mostly camera movement. Trust patterns across seeds, not single numbers, and always check the videos by eye.